# Global Recommendations Demo — Hindi Songs for an English Playlist

The contrastive encoder maps *any* song's audio features into the learned
embedding space, regardless of language. This demo embeds a separately curated
set of 1,801 popular Hindi songs and recommends the ones closest (by cosine
similarity) to the mean embedding of an English playlist.

Requires a trained model (`python -m src.contrastive`).

In [1]:
import os
import sys

# Run from the repository root so config's relative paths resolve
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd

from src import config
from src.contrastive import (load_song_features, load_trained_model,
                             precompute_embeddings, recommend_songs)

## Load the trained encoder and embed both song libraries

In [2]:
df_features = load_song_features()
model = load_trained_model(df_features)

english_embeddings = precompute_embeddings(model, df_features)
english_embeddings.shape

(13835, 17)

In [3]:
hindi_df = pd.read_csv(config.HINDI_FEATURES_PATH).drop_duplicates(subset='unique_track')
hindi_features = pd.concat([hindi_df['unique_track'],
                            hindi_df.drop(['id', 'unique_track'], axis=1)], axis=1)

hindi_embeddings = precompute_embeddings(model, hindi_features)
hindi_embeddings.shape

(1196, 17)

## Pick an English playlist

In [4]:
# An example playlist — swap in any songs from data/audio_features.csv
playlist = [
    'Coldplay - Fix You',
    'Coldplay - The Scientist',
    'Ed Sheeran - Thinking Out Loud',
    'Adele - Someone Like You',
    'John Mayer - Gravity',
    'Bon Iver - Skinny Love',
    'The Lumineers - Ho Hey',
    'Of Monsters and Men - Little Talks',
    'Vance Joy - Riptide',
    'Passenger - Let Her Go',
]

## Recommend Hindi songs for it

In [5]:
recommendations = recommend_songs(english_embeddings, hindi_embeddings, playlist)

pd.DataFrame(recommendations[:20], columns=['Hindi song', 'Cosine similarity'])

,Hindi song,Cosine similarity
0,Tum Hi Aana (Sad Version),0.955379
1,Baarishein,0.953521
2,Ek Hi Khwab Kai Baar Dekha,0.944031
3,Pehli Sharam,0.943332
4,Tujhko Jo Paaya,0.942539
5,Channa Mereya - Unplugged,0.938664
6,Palat Meri Jaan,0.937483
7,Maiyya Mainu,0.931086
8,Kaise Hua,0.929379
9,Roz Roz,0.928781


The same call works for any target library: pass any set of songs with
Spotify audio features as the second argument of `recommend_songs`, and the
model recommends from that set — the encoder itself is language-agnostic.